# Behind the pipeline

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


### An example of pipeline

In [ ]:
from transformers import pipeline

classifier = pipeline('sentiment-analysis')
classifier([
    "This is a sunny day.",
    "I don't like pudding."
])

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9997976422309875},
 {'label': 'NEGATIVE', 'score': 0.9976029992103577}]

## How everything works

Pipeline groups together 3 tasks:
* Preprocessing
* Passing the inputs through the model
* Postprocessing

### Preprocessing with a tokenizer

* First step is to convert the text inputs into numbers that the model can understand.
* We use `tokenizer` for this, which is responsible for:
    * Spliting the input into `tokens`
    * Mapping each token to numbers
    * Adding additional inputs that may be useful to the model
* Preprocessing should be done in the same way the model was pretrained
* For doing this we use the `AutoTokenizer` class and its `from_pretrained` method
* Using the checkpoint name of the model, it will automatically fetch the data associated to the model's tokenizer and cache it

In [ ]:
from transformers import AutoTokenizer
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

* We can now pass the text inputs directly to the tokenizer and get back a dict
* Only work left is to convert the input ids into tensors
* **Transformer models only accept `tensors` as input**
* To specify the type of tensors we want to get back (PyTorch or plain NumPy), we use the return_tensors argument
* We can pass a sentence or a list of sentences, as well as specifying the type of tensors we want to get back (if no type is passed, we will get a list of lists as a result)

In [ ]:
raw_inputs = [
    "This is a sunny day.",
    "I don't like pudding."
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors='pt')
print(inputs)

{'input_ids': tensor([[  101,  2023,  2003,  1037, 11559,  2154,  1012,   102,     0],
        [  101,  1045,  2123,  1005,  1056,  2066, 29593,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])}


The output is dictionary containing 3 keys:
1. `input_ids`: Contains unique ids of tokens in each sentence
2. `token_type_ids`
3. `attention_mask`: marks which tokens are paddings(0) vs real(1)

But out main point of interest are `input_ids` and `attention_mask`. The third key appears for models that use **segment embeddings**

### Going through the model

* We can download the model the same we did with the tokenizer
* We use the `AutoModel` class with its `from_pretrained` method


In [ ]:
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.bias       | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


* This achitecture contains only the base transformer module: given some inputs, it outputs *hidden states* also called **features**
*  For each model input, we’ll retrieve a **high-dimensional vector representing the contextual understanding of that input by the Transformer model**
* These hidden states can be useful on their own, but usually they are inputs to another part of the model called `head`

#### High-dimensional vector

The vector output by the model is large and generally has 3 dimensions:
1. **Batch Size**: The number of sequences processed at a time
2. **Sequence Length**: The length of the numerical representation of the sequences
3. **Hidden size**: The vector dimension of each model input

It is called high-dimensional because of `hidden size` as it can be very large

In [ ]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([2, 9, 768])


#### Model Head

* The model head take the high dimensional vector of hidden states as inputs and project them onto a differen dimension.
* They usually are composed of one or more linear layers.
* The output of the model is directly sent to the model head to be processed.
* For out example we need a model with a sequence classification head.
* Therefore instead of using the `AutoModel` class we will use the `AutoModelForSequenceClassification` class

In [ ]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
outputs.logits.shape

torch.Size([2, 2])

Since we have just two sentences and two labels, the result we get from our model is of shape 2 x 2.

### Postprocessing the output

In [ ]:
outputs.logits

tensor([[-4.0830,  4.4222],
        [ 3.3011, -2.7300]], grad_fn=<AddmmBackward0>)

* These are `logits`, the raw unnormalized scores outputted by the last layer of the model.
* To be converted to probabilities they need to go throught a `Softmax` layer.

In [ ]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
predictions

tensor([[2.0237e-04, 9.9980e-01],
        [9.9760e-01, 2.3970e-03]], grad_fn=<SoftmaxBackward0>)

To get the labels corresponding to each position, we can inspect the id2label attribute of the model config

In [ ]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

We have successfully reproduced the pipeline.

# Models
Creating and using models

## Creating a Transformer

* The `AutoModel` class and its associates are actually simple wrappers designed to fetch appropriate model architecture for a given checkpoint.
* If we know the type of model we want to use, we can simply use the class that defines its architecture.

In [ ]:
# Using AutoModel class
from transformers import AutoModel
model = AutoModel.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Using the class of the model
from transformers import BertModel
model = BertModel.from_pretrained("bert-base-cased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Loading and saving

The models have a `save_pretrained()` method, which saves the model weights and achitecture.

In [ ]:
model.save_pretrained("model_state")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

This creates 2 files in the directory `model_state`
* `config.json`: This contains all the necessary attributes needed to build the model architecture. This also contains some metadata.
* `model.safetensors`: This is known as the state dictionary; it contains all the model's weights.

The two files work together: the configuration file is needed to know about the model architecture, while the model weights are the parameters of the model.

In [ ]:
# To reuse the saved model used `from_pretrained()`
from transformers import AutoModel
model = AutoModel.from_pretrained("model_state")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Encoding text

Transformer models handle text by conveting them into numbers as we have already seen.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

encoded_input = tokenizer("Hello, I'm a single sentence!")
print(encoded_input)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

{'input_ids': [101, 8667, 117, 146, 112, 182, 170, 1423, 5650, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


We get a dictionary with the following fields:

* **input_ids**: numerical representations of our tokens
* **token_type_id**s: these tell the model which part of the input is sentence A and which is sentence B
* **attention_mask**: this indicates which tokens should be attended to and which should not

We can decode the input ids to get back the text

In [ ]:
tokenizer.decode(encoded_input['input_ids'])

"[CLS] Hello, I ' m a single sentence! [SEP]"

`[CLS]` adn `[SEP]` are special tokens and not all models require them.

We can encode multiple sentences at once, either by batching them together or by passing a list

In [ ]:
encoded_input = tokenizer(["How are you?", "I'm fine, thank you!"],padding=True ,truncation=True , return_tensors='pt')
encoded_input

{'input_ids': tensor([[ 101, 1731, 1132, 1128,  136,  102,    0,    0,    0,    0],
        [ 101,  146,  112,  182, 2503,  117, 6243, 1128,  106,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

### Padding inputs
If we ask the tokenizer to pad the inputs, it will make all sentences the same length by adding a special padding token to the sentences that are shorter than the longest one.

The padding tokens have an `attention_mask value` = 0. These padding tokens shouldn't be analysed by the model as they are not part of the sentence

### Truncating inputs
The tensors might get too big to be processed by the model. For instance, BERT was only pretrained with sequences up to 512 tokens, so it cannot process longer sequences. If we have sequences longer than the model can handle, we’ll need to truncate them with the `truncation` parameter

# Tokenizers

Tokenizers as only one purpose -> to translate text into numbers that can be processed by the model.

The goal is to find the most meaningful representation - that is, the one that makes most sense to the model - and, if possible, the smallest representation.

## Word-based

* Very easy to setup and few rules and it often yiedls decent results.
* There are different ways to split the text
    * Use whitespace to tokenize the text into words using the `split()` function

In [1]:
tokenize_text = "Today is a rainy day.".split()
tokenize_text

['Today', 'is', 'a', 'rainy', 'day.']

**Vocabulary** -> is defined by total number of independent tokens in our corpus.

Each word gets assigned an ID, starting from 0 and going up to the size of the vocabulary. The model uses these IDs to identify each word.

* We need a custom token to represent words that are not in our vocabulary. This is known as the “unknown” token, often represented as ”[UNK]”.
* It is a bad sign if a tokenizer is producing a lot of these.
* The goal when crafting the vocabulary is to do it in such a way that the tokenizer tokenizes as few words as possible into the unknown token.


One way to reduce the number of unknown tokens is to go one-level deeper i.e. `character based` tokenizer.

## Character-based

Character-based tokenizer splits the text into characters. This provides:
* A smaller vocabulary
* Less number of unknown tokens, as words are built from characters

But here too question arises concerning spaces and punctuation

* Since, the representation is now character based they are less meaningfull, but this can differ from language to language.
* The overall number of tokens becomes very large.

## Subword Tokenization

Subword tokenization algorithms rely on the principle that frequently used words should not be split into smaller subwords, but rare words should be decomposed into meaningful subwords.

These subwords endup providing a lot of semantic meaning.

### More

There a more tokenizers like:
* Byte-level, BPE like in GPT-2
* WordPiece, used in BERT
* SentencePiece or Unigram, as used in several multilingual models

## Loading and Saving

In [2]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-cased")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [3]:
# Using the AutoTokenizer class
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [4]:
tokenizer("Using a Transformer network is simple")

{'input_ids': [101, 7993, 170, 13809, 23763, 2443, 1110, 3014, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

Saving a tokenizer is identical to saving a model

In [5]:
tokenizer.save_pretrained("tokenizer")

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

## Encoding

Translating text into numbers is called **encoding**.

Encoding is a 2 steps process:
1. Tokenizing
2. Conversion to Input IDs

We build a tensor our of the tokens and feed it into the model.

To do this, the tokenizer has a vocabulary, which is the part we download when we instantiate it with the from_pretrained() method.

### Tokenization

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

sequence = "Using a Transformer network is simple"
tokens = tokenizer.tokenize(sequence)

print(tokens)

['Using', 'a', 'Trans', '##former', 'network', 'is', 'simple']


### Tokens to Input IDs

In [7]:
ids = tokenizer.convert_tokens_to_ids(tokens)

print(ids)

[7993, 170, 13809, 23763, 2443, 1110, 3014]


## Decoding

It is the reverse of encoding. This can be done by using the `decode()` method

In [8]:
decoded_string = tokenizer.decode([7993, 170, 11303, 1200, 2443, 1110, 3014])
print(decoded_string)

Using a transformer network is simple
